<a href="https://colab.research.google.com/github/lovey7768/scalable-async-task-engine/blob/main/scalable_async_task_engine.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cell 1: Install PostgreSQL & Redis Services on Colab

In [6]:
# 1.install PostgreSQL and Redis server on the colab linux system
!apt-get update -qq
!apt-get install -y -qq postgresql redis-server

# 2.start the database and queue services in the background
# Using pg_ctlcluster for a more explicit PostgreSQL start
!pg_ctlcluster 14 main start
!service redis-server start

# Verify PostgreSQL status
import time
time.sleep(2) # Give PostgreSQL a moment to fully start
!pg_isready -h localhost -p 5432

# 3.Create a dedicated database user and database
!sudo -u postgres psql -c "CREATE USER appuser WITH PASSWORD 'apppassword';"
!sudo -u postgres psql -c "CREATE DATABASE microservice_db OWNER appuser;"
print("PostgreSQL & Redis are installed and running in the background!")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package libjemalloc2:amd64.
(Reading database ... 118332 files and directories currently installed.)
Preparing to unpack .../00-libjemalloc2_5.2.1-4ubuntu1_amd64.deb ...
Unpacking libjemalloc2:amd64 (5.2.1-4ubuntu1) ...
Selecting previously unselected package liblua5.1-0:amd64.
Preparing to unpack .../01-liblua5.1-0_5.1.5-8.1build4_amd64.deb ...
Unpacking liblua5.1-0:amd64 (5.1.5-8.1build4) ...
Selecting previously unselected package liblzf1:amd64.
Preparing to unpack .../02-liblzf1_3.6-3_amd64.deb ...
Unpacking liblzf1:amd64 (3.6-3) ...
Selecting previously unselected package lua-bitop:amd64.
Preparing to unpack .../03-lua-bitop_1.0.2-5_amd64.deb ...
Unpacking lua-bitop:amd64 (1.0.2-5) ...
Selecting previously unselected package lua-cjson:a

# Cell 2: Install Python Libraries

In [7]:
!pip install -q fastapi uvicorn sqlalchemy asyncpg redis pydantic-settings httpx nest_asyncio pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 560.6/560.6 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.9 MB/s eta 0:00:00


fastapi & uvicorn: The web framework and web server.

sqlalchemy & asyncpg: High-speed async database toolkit for PostgreSQL.

redis: Connects Python to the Redis queue.

pydantic-settings: Validates settings and configuration.

# Cell 3: Create the Project Folders

In [8]:
import os
os.makedirs("app/core",exist_ok=True)
os.makedirs("app/models",exist_ok=True)
os.makedirs("app/schemas",exist_ok=True)
os.makedirs("app/services",exist_ok=True)
os.makedirs("app/workers",exist_ok=True)
os.makedirs("app/api",exist_ok=True)

print("Project directories created!")

Project directories created!


# Cell 4: App Configuration (app/core/config.py)
This file holds your passwords, database URLs, and port numbers in one safe place:

In [9]:
%%writefile app/core/config.py
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
  PROJECT_NAME: str= "Colab Async AI Microservice"
  API_V1_STR: str="/api/v1"

  # local connection to postgreSQL inside Colab
  POSTGRES_USER: str= "appuser"
  POSTGRES_PASSWORD: str= "apppassword"
  POSTGRES_SERVER: str="localhost"
  POSTGRES_PORT: int=5432
  POSTGRES_DB: str="microservice_db"

  # local connection to Redis inside colab
  REDIS_HOST: str="localhost"
  REDIS_PORT: int=6379

  @property
  def ASYNC_DATABASE_URI(self) -> str:
    return f"postgresql+asyncpg://{self.POSTGRES_USER}:{self.POSTGRES_PASSWORD}@{self.POSTGRES_SERVER}:{self.POSTGRES_PORT}/{self.POSTGRES_DB}"
settings= Settings()

Writing app/core/config.py


# Cell 5: Database Connection & Tables (app/core/database.py & app/models/task.py)
Database Engine:

In [10]:
%%writefile app/core/database.py
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from sqlalchemy.orm import DeclarativeBase
from app.core.config import settings

# create connection pool to PostgreSQL
engine=create_async_engine(settings.ASYNC_DATABASE_URI, echo=False)
AsyncSessionLocal= async_sessionmaker(bind=engine, class_=AsyncSession,expire_on_commit=False)

class Base(DeclarativeBase):
  pass
async def get_db():
  async with AsyncSessionLocal() as session:
    yield session

Writing app/core/database.py


# Database Schema:

In [11]:
%%writefile app/models/task.py
import uuid
import enum
from datetime import datetime
from sqlalchemy import String, Text, Enum, DateTime, JSON
from sqlalchemy.dialects.postgresql import UUID
from sqlalchemy.orm import Mapped, mapped_column
from app.core.database import Base

class TaskStatus(str, enum.Enum):
    PENDING = "PENDING"
    PROCESSING = "PROCESSING"
    COMPLETED = "COMPLETED"
    FAILED = "FAILED"

class Task(Base):
    __tablename__ = "tasks"

    id: Mapped[uuid.UUID] = mapped_column(UUID(as_uuid=True), primary_key=True, default=uuid.uuid4)
    task_type: Mapped[str] = mapped_column(String(50), nullable=False)
    input_payload: Mapped[str] = mapped_column(Text, nullable=False)
    status: Mapped[TaskStatus] = mapped_column(Enum(TaskStatus), default=TaskStatus.PENDING)
    result: Mapped[dict | None] = mapped_column(JSON, nullable=True)
    created_at: Mapped[datetime] = mapped_column(DateTime, default=datetime.utcnow)

Writing app/models/task.py


# Cell 6: Data Validation Schemas (app/schemas/task.py)
Pydantic checks incoming requests to ensure users don't send corrupt or empty data:

In [12]:
%%writefile app/schemas/task.py
import uuid
from datetime import datetime
from pydantic import BaseModel, Field
from app.models.task import TaskStatus

class TaskCreateRequest(BaseModel):
  task_type: str= Field(..., example="text_analysis")
  input_payload: str = Field(...,min_length=3, example="FastAPI with PostgreSQL and Redis makes microservices blazing fast.")

class TaskResponse(BaseModel):
  id: uuid.UUID
  task_type: str
  status: TaskStatus
  result: dict | None=None
  created_at: datetime

  class Config:
    from_attributes = True

Writing app/schemas/task.py


# Cell 7: AI Engine & Background Worker (app/services/ & app/workers/)
The AI Logic:

In [13]:
%%writefile app/services/ai_engine.py
import asyncio

class AIEngine:
    @staticmethod
    async def analyze_text(text: str) -> dict:
        # Simulate neural computation time (2 seconds)
        await asyncio.sleep(2.0)

        words = text.split()
        keywords = [w.lower() for w in words if len(w) > 4]
        sentiment = "Positive" if any(w in ["fast", "great", "scalable", "blazing", "powerful"] for w in words) else "Neutral"

        return {
            "word_count": len(words),
            "character_count": len(text),
            "sentiment": sentiment,
            "detected_keywords": list(set(keywords))[:5]
        }

Writing app/services/ai_engine.py


# The Background Worker

In [14]:
%%writefile app/workers/background_worker.py
import asyncio
import uuid
import redis.asyncio as aioredis
from sqlalchemy import select
from app.core.config import settings
from app.core.database import AsyncSessionLocal
from app.models.task import Task, TaskStatus
from app.services.ai_engine import AIEngine

async def worker_loop():
    r = aioredis.Redis(host=settings.REDIS_HOST, port=settings.REDIS_PORT, decode_responses=True)
    print(" Worker is ready and waiting for jobs in the Redis queue...")

    while True:
        try:
            # Wait for a new task ID to be added to the queue
            task_item = await r.brpop("task_queue", timeout=3)
            if not task_item:
                continue

            _, task_id_str = task_item
            task_id = uuid.UUID(task_id_str)
            print(f" [Worker] Picked up Task: {task_id}")

            async with AsyncSessionLocal() as session:
                query = select(Task).where(Task.id == task_id)
                res = await session.execute(query)
                task = res.scalar_one_or_none()

                if task:
                    task.status = TaskStatus.PROCESSING
                    await session.commit()

                    # Run AI computation
                    result = await AIEngine.analyze_text(task.input_payload)

                    task.result = result
                    task.status = TaskStatus.COMPLETED
                    await session.commit()
                    print(f" [Worker] Task {task_id} COMPLETED and saved to PostgreSQL!")

        except Exception as e:
            print(f" Worker Error: {e}")
            await asyncio.sleep(1)

if __name__ == "__main__":
    asyncio.run(worker_loop())

Writing app/workers/background_worker.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Cell 8: FastAPI API Endpoints & Main App (app/main.py)

In [15]:
%%writefile app/main.py
import uuid
from contextlib import asynccontextmanager
from fastapi import FastAPI, Depends, HTTPException, status
from sqlalchemy.ext.asyncio import AsyncSession
from sqlalchemy import select
import redis.asyncio as aioredis

from app.core.config import settings
from app.core.database import engine, Base, get_db
from app.models.task import Task, TaskStatus
from app.schemas.task import TaskCreateRequest, TaskResponse

@asynccontextmanager
async def lifespan(app: FastAPI):
    # Automatically create the PostgreSQL tables on startup
    async with engine.begin() as conn:
        await conn.run_sync(Base.metadata.create_all)
    yield
    await engine.dispose()

app = FastAPI(title=settings.PROJECT_NAME, lifespan=lifespan)

async def get_redis():
    client = aioredis.Redis(host=settings.REDIS_HOST, port=settings.REDIS_PORT, decode_responses=True)
    try:
        yield client
    finally:
        await client.close()

@app.post("/api/v1/tasks", response_model=TaskResponse, status_code=status.HTTP_202_ACCEPTED)
async def submit_task(
    payload: TaskCreateRequest,
    db: AsyncSession = Depends(get_db),
    r: aioredis.Redis = Depends(get_redis)
):
    # 1. Save new task with PENDING status in PostgreSQL
    new_task = Task(task_type=payload.task_type, input_payload=payload.input_payload)
    db.add(new_task)
    await db.commit()
    await db.refresh(new_task)

    # 2. Push task ID into the Redis queue for the background worker
    await r.lpush("task_queue", str(new_task.id))
    return new_task

@app.get("/api/v1/tasks/{task_id}", response_model=TaskResponse)
async def check_task_status(task_id: uuid.UUID, db: AsyncSession = Depends(get_db)):
    query = select(Task).where(Task.id == task_id)
    res = await db.execute(query)
    task = res.scalar_one_or_none()

    if not task:
        raise HTTPException(status_code=404, detail="Task not found")
    return task

@app.get("/health")
def health():
    return {"status": "healthy", "architecture": "FastAPI + Postgres + Redis"}

Writing app/main.py


# Cell 9: Start the Background Worker and API Server

In [16]:
import subprocess
import time
import os

# Terminate any previously running processes to ensure a clean start
print("Attempting to terminate old uvicorn and worker processes...")
os.system("pkill -f uvicorn")
os.system("pkill -f 'python -m app.workers.background_worker'")
time.sleep(2) # Give processes a moment to terminate
print("Old processes termination attempt complete.")

# 1. Start the Background Worker process
worker_process = subprocess.Popen(["python", "-m", "app.workers.background_worker"])
print("1️ Background Worker Process Started!")

# --- Diagnostic Step: Print content of app/core/database.py ---
print("\n--- Content of app/core/database.py before Uvicorn start ---")
with open('app/core/database.py', 'r') as f:
    print(f.read())
print("-----------------------------------------------------------\n")
# ---------------------------------------------------------------

# 2. Start the FastAPI Uvicorn Server on port 8000
# Capture stdout and stderr for better debugging
api_process = subprocess.Popen(
    ["uvicorn", "app.main:app", "--host", "127.0.0.1", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)
print("2️ FastAPI Server Initiated on http://127.0.0.1:8000 (Checking status...)")

# Wait a bit longer for boot
time.sleep(7)

# Check if uvicorn process is still running
if api_process.poll() is not None:
    print(" Uvicorn process exited prematurely!")
    print("Uvicorn Stdout:", api_process.stdout.read())
    print("Uvicorn Stderr:", api_process.stderr.read())
else:
    print(" Uvicorn process is still running.")

# Verify FastAPI server response locally using curl
print("\nVerifying FastAPI server response locally with curl...")
try:
    result = subprocess.run(["curl", "-s", "--fail", "http://127.0.0.1:8000/health"], capture_output=True, text=True)
    if result.returncode == 0:
        print(" FastAPI /health endpoint returned success (exit code 0). Server seems active.")
    else:
        print(f" FastAPI /health endpoint returned an error (exit code {result.returncode}). Server might not be fully ready or healthy. Output: {result.stdout.strip()}, Error: {result.stderr.strip()}")
except Exception as e:
    print(f" Error checking FastAPI server with curl: {e}")

print("Proceeding with tests...")

Attempting to terminate old uvicorn and worker processes...
Old processes termination attempt complete.
1️ Background Worker Process Started!

--- Content of app/core/database.py before Uvicorn start ---
from sqlalchemy.ext.asyncio import create_async_engine, async_sessionmaker, AsyncSession
from sqlalchemy.orm import DeclarativeBase
from app.core.config import settings

# create connection pool to PostgreSQL
engine=create_async_engine(settings.ASYNC_DATABASE_URI, echo=False)
AsyncSessionLocal= async_sessionmaker(bind=engine, class_=AsyncSession,expire_on_commit=False)

class Base(DeclarativeBase):
  pass
async def get_db():
  async with AsyncSessionLocal() as session:
    yield session

-----------------------------------------------------------

2️ FastAPI Server Initiated on http://127.0.0.1:8000 (Checking status...)
 Uvicorn process is still running.

Verifying FastAPI server response locally with curl...
 FastAPI /health endpoint returned success (exit code 0). Server seems active

# Cell 10: Test the Live System (Interactive Verification)
Let's simulate a user submitting a task and watching it complete asynchronously!

In [17]:
import httpx
import time

async def test_microservice():
    async with httpx.AsyncClient(base_url="http://127.0.0.1:8000") as client:
        # Step 1: Health check
        res = await client.get("/health")
        print(" Health Check:", res.json())

        # Step 2: Submit a heavy AI job
        print("\n Submitting new text analysis job...")
        payload = {
            "task_type": "text_analysis",
            "input_payload": "Building asynchronous microservices with FastAPI, Redis, and PostgreSQL creates scalable architectures."
        }
        create_res = await client.post("/api/v1/tasks", json=payload)
        task_data = create_res.json()
        task_id = task_data["id"]
        print(f" Immediate Ticket Returned! ID: {task_id} | Initial Status: {task_data['status']}")

        # Step 3: Poll the status while the background worker processes it
        print("\n Polling server to check task completion...")
        for i in range(5):
            time.sleep(1)
            status_res = await client.get(f"/api/v1/tasks/{task_id}")
            current_status = status_res.json()
            print(f"⏱ Check #{i+1}: Status = {current_status['status']}")

            if current_status["status"] == "COMPLETED":
                print("\n Task Finished! Full Result from Database:")
                import json
                print(json.dumps(current_status, indent=2))
                break

# Run the test inside Colab
import asyncio
import nest_asyncio
nest_asyncio.apply()
asyncio.run(test_microservice())

 Health Check: {'status': 'healthy', 'architecture': 'FastAPI + Postgres + Redis'}

 Submitting new text analysis job...
 Immediate Ticket Returned! ID: 2e966f47-7be1-4500-ae57-f4f5d10d6342 | Initial Status: PENDING

 Polling server to check task completion...
⏱ Check #1: Status = PROCESSING
⏱ Check #2: Status = PROCESSING
⏱ Check #3: Status = COMPLETED

 Task Finished! Full Result from Database:
{
  "id": "2e966f47-7be1-4500-ae57-f4f5d10d6342",
  "task_type": "text_analysis",
  "status": "COMPLETED",
  "result": {
    "word_count": 11,
    "character_count": 103,
    "sentiment": "Positive",
    "detected_keywords": [
      "creates",
      "redis,",
      "postgresql",
      "architectures.",
      "asynchronous"
    ]
  },
  "created_at": "2026-08-15T10:38:53.341556"
}


# Cell 11: Inspect the Raw PostgreSQL Database Rows
To prove the data is permanently stored in PostgreSQL, query it directly:

In [18]:
!PGPASSWORD=apppassword /usr/lib/postgresql/14/bin/psql -U appuser -h localhost -p 5432 -d microservice_db -c "SELECT id, task_type, status, result, created_at FROM tasks;"

/usr/lib/postgresql/14/bin/psql
